# 06 Curves and dose-response: NIST nonlinear regression, Puromycin, Bliss's beetles

- `table.fit_formula` (nonlinear least squares) on the 27 **NIST StRD nonlinear regression** problems, each from both
  of NIST's starting points: start 1 (far from the answer) and start 2 (closer). NIST certifies the parameters, their
  standard errors and the residual standard deviation to 11 digits.
- **Puromycin** (Treloar 1974): a Michaelis-Menten curve, whose fit Bates & Watts (1988) printed.
- **Bliss's beetles** (Bliss 1935): a dose-response curve for yes/no outcomes out of a number of beetles, with
  `table.explain(outcome="yes or no", out_of=)`, whose fit Dobson & Barnett (2008) printed, and the dose that kills
  half with `table.fit_dose_response`.

**Only simplydata runs in this notebook.** The reference values are in `references/06_curves_and_dose_response.csv`,
written by `scripts/references/06_curves_and_dose_response.py`: the NIST certified values (a nonlinear fit must agree
to at least 6 significant digits, LRE, the level of simplydata's own reliability study, after McCullough 1999), the
printed fits (half a unit of the last printed digit), and R `glm()` for the LD50 to more digits (relative difference
1e-6). Each check prints its source.

In [1]:
import simplydata as sdata
from simplydata import SimplydataError

from checks import ReferenceChecks, data_file, quiet, read_row, read_rows, read_rows_by, versions

NOTEBOOK = "06_curves_and_dose_response"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit 3f0192b of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## The NIST problems

The formulas as NIST gives them, in simplydata's syntax (`**` for powers). Roszman1 needs `arctan`, which
`fit_formula` gained after 0.2.0 (FINDINGS A3). Nelson's model is for log(y), so the data have a column `log_y`.
NIST's two starting values for each parameter are read from a small table next to the data.

In [2]:
RATIONAL3 = "(b1 + b2*x + b3*x**2 + b4*x**3) / (1 + b5*x + b6*x**2 + b7*x**3)"
NIST_FORMULAS = {
    # lower difficulty
    "Misra1a": "y = b1*(1-exp(-b2*x))",
    "Chwirut2": "y = exp(-b1*x)/(b2+b3*x)",
    "Chwirut1": "y = exp(-b1*x)/(b2+b3*x)",
    "Lanczos3": "y = b1*exp(-b2*x) + b3*exp(-b4*x) + b5*exp(-b6*x)",
    "Gauss1": "y = b1*exp(-b2*x) + b3*exp(-(x-b4)**2/b5**2) + b6*exp(-(x-b7)**2/b8**2)",
    "Gauss2": "y = b1*exp(-b2*x) + b3*exp(-(x-b4)**2/b5**2) + b6*exp(-(x-b7)**2/b8**2)",
    "DanWood": "y = b1*x**b2",
    "Misra1b": "y = b1*(1-(1+b2*x/2)**(-2))",
    # average difficulty
    "Kirby2": "y = (b1 + b2*x + b3*x**2) / (1 + b4*x + b5*x**2)",
    "Hahn1": "y = " + RATIONAL3,
    "Nelson": "log_y = b1 - b2*x1*exp(-b3*x2)",
    "MGH17": "y = b1 + b2*exp(-x*b4) + b3*exp(-x*b5)",
    "Lanczos1": "y = b1*exp(-b2*x) + b3*exp(-b4*x) + b5*exp(-b6*x)",
    "Lanczos2": "y = b1*exp(-b2*x) + b3*exp(-b4*x) + b5*exp(-b6*x)",
    "Gauss3": "y = b1*exp(-b2*x) + b3*exp(-(x-b4)**2/b5**2) + b6*exp(-(x-b7)**2/b8**2)",
    "Misra1c": "y = b1*(1-(1+2*b2*x)**(-0.5))",
    "Misra1d": "y = b1*b2*x*((1+b2*x)**(-1))",
    "Roszman1": "y = b1 - b2*x - arctan(b3/(x-b4))/pi",
    "ENSO": "y = b1 + b2*cos(2*pi*x/12) + b3*sin(2*pi*x/12) + b5*cos(2*pi*x/b4) + b6*sin(2*pi*x/b4)"
            " + b8*cos(2*pi*x/b7) + b9*sin(2*pi*x/b7)",
    # higher difficulty
    "MGH09": "y = b1*(x**2+x*b2) / (x**2+x*b3+b4)",
    "Thurber": "y = " + RATIONAL3,
    "BoxBOD": "y = b1*(1-exp(-b2*x))",
    "Rat42": "y = b1 / (1+exp(b2-b3*x))",
    "MGH10": "y = b1 * exp(b2/(x+b3))",
    "Eckerle4": "y = (b1/b2) * exp(-0.5*((x-b3)/b2)**2)",
    "Rat43": "y = b1 / ((1+exp(b2-b3*x))**(1/b4))",
    "Bennett5": "y = b1 * (b2+x)**(-1/b3)",
}
load("nist_Misra1a").fit_formula(NIST_FORMULAS["Misra1a"], start={"b1": 500, "b2": 0.0001})

Fitted "y = b1*(1-exp(-b2*x))" to 14 rows: b1 = 239 (95 % CI 233 to 245), b2 = 0.000550 (95 % CI 0.000534 to 0.000566). Residual standard error 0.102 on 12 degrees of freedom; R squared = 1.00 (only a rough guide for a curve).
Values found: b1, b2. Data: x. Method: nonlinear least squares (Levenberg-Marquardt), as R nls(); intervals are estimate +/- t x standard error (R confint() on nls profiles the likelihood and can differ a little).


,test,formula,b1,standard error of b1,t of b1,p-value of b1,95 % CI low of b1,95 % CI high of b1,b2,standard error of b2,...,95 % CI low of b2,95 % CI high of b2,residual standard error,degrees of freedom,R squared,AIC,rows,converged,method,note
1,formula fit (nonlinear least squares),y = b1*(1-exp(-b2*x)),238.942129,2.707008,88.267993,2.985634e-18,233.044066,244.840192,0.00055,0.000007,...,0.000534,0.000566,0.101879,12.0,0.999982,-20.37904,14,yes,Levenberg-Marquardt,converged


In [3]:
for name, formula in NIST_FORMULAS.items():
    data = load(f"nist_{name}")
    starts = read_rows(load(f"nist_{name}_starts"))
    for start_number in (1, 2):
        start = {row["parameter"]: row[f"start {start_number}"] for row in starts}
        try:
            found = read_row(quiet(data.fit_formula, formula, start=start))
        except SimplydataError as refused:
            print(f"simplydata gave no fit for {name} from start {start_number}: {str(refused).splitlines()[0]}")
            found = {}
        values = {}
        for b in start:
            values[b] = found.get(b)
            values[f"{b} standard error"] = found.get(f"standard error of {b}")
        values["residual standard deviation"] = found.get("residual standard error")
        checks.check_values(f"NIST {name}", f"fit_formula (start {start_number})", values)

agrees  NIST Misra1a | fit_formula (start 1) | b1: simplydata 238.942129288, NIST certified 238.94212918; LRE 9.3 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1a.shtml
agrees  NIST Misra1a | fit_formula (start 1) | b1 standard error: simplydata 2.70700762418, NIST certified 2.7070075241; LRE 7.4 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1a.shtml
agrees  NIST Misra1a | fit_formula (start 1) | b2: simplydata 0.000550156431513, NIST certified 0.00055015643181; LRE 9.3 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1a.shtml
agrees  NIST Misra1a | fit_formula (start 1) | b2 standard error: simplydata 7.26686915491e-06, NIST certified 7.2668688436e-06; LRE 7.4 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1a.shtml
agrees  NIST Misra1a | fit_formula

agrees  NIST MGH17 | fit_formula (start 1) | b1: simplydata 0.37541005286, NIST certified 0.37541005211; LRE 8.7 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/mgh17.shtml
agrees  NIST MGH17 | fit_formula (start 1) | b1 standard error: simplydata 0.00207231539227, NIST certified 0.0020723153551; LRE 7.7 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/mgh17.shtml
agrees  NIST MGH17 | fit_formula (start 1) | b2: simplydata 1.93584697041, NIST certified 1.9358469127; LRE 7.5 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/mgh17.shtml
agrees  NIST MGH17 | fit_formula (start 1) | b2 standard error: simplydata 0.220316545972, NIST certified 0.22031669222; LRE 6.2 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/mgh17.shtml
agrees  NIST MGH17 | fit_formula (start 1) | b3: simplyda

agrees  NIST Lanczos1 | fit_formula (start 2) | b1: simplydata 0.0951000000274, NIST certified 0.095100000027; LRE 11.4 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/lanczos1.shtml
explained difference – precision limit (B1)  NIST Lanczos1 | fit_formula (start 2) | b1 standard error: simplydata 5.33317525771e-11, NIST certified 5.3347304234e-11; LRE 3.5 (needs >= 6)
      why: NIST nonlinear problems: some estimates and standard errors agree to 3 to 6 digits - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#b1-nist-nonlinear-problems-some-estimates-and-standard-errors-agree-to-3-to-6-digits
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/lanczos1.shtml
agrees  NIST Lanczos1 | fit_formula (start 2) | b2: simplydata 1.00000000013, NIST certified 1.0000000001; LRE 10.6 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/

agrees  NIST MGH10 | fit_formula (start 1) | b1: simplydata 0.00560963695483, NIST certified 0.005609636471; LRE 7.1 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/mgh10.shtml
explained difference – machine (B1)  NIST MGH10 | fit_formula (start 1) | b1 standard error: simplydata 0.000156879222922, NIST certified 0.00015687892471; LRE 5.7 (needs >= 6)
      why: NIST nonlinear problems: some estimates and standard errors agree to 3 to 6 digits - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#b1-nist-nonlinear-problems-some-estimates-and-standard-errors-agree-to-3-to-6-digits
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/mgh10.shtml
agrees  NIST MGH10 | fit_formula (start 1) | b2: simplydata 6181.34627438, NIST certified 6181.3463463; LRE 7.9 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/mgh10.shtml
explained di

agrees  NIST Eckerle4 | fit_formula (start 1) | b1: simplydata 1.55438271783, NIST certified 1.5543827178; LRE 10.8 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/eckerle4.shtml
agrees  NIST Eckerle4 | fit_formula (start 1) | b1 standard error: simplydata 0.0154080512233, NIST certified 0.015408051163; LRE 8.4 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/eckerle4.shtml
agrees  NIST Eckerle4 | fit_formula (start 1) | b2: simplydata 4.088832176, NIST certified 4.0888321754; LRE 9.8 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/eckerle4.shtml
agrees  NIST Eckerle4 | fit_formula (start 1) | b2 standard error: simplydata 0.0468030215894, NIST certified 0.046803020753; LRE 7.7 (needs >= 6)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/nls/data/eckerle4.shtml
agrees  NIST Eckerle4 | fit_formula

### BoxBOD from NIST's far start: does simplydata say the fit failed?

From start 1 (b1 = 1, b2 = 1) SciPy `curve_fit`, which `fit_formula` calls, stops in a flat region far from the
certified values, so the estimates above FAIL against NIST (FINDINGS B5). simplydata 0.2.0 still reported
`converged = "yes"` there (FINDINGS A2). What matters for a user is that simplydata says so: it must report
`converged = "no"` (recorded as 1 against the expected 1) and print a note that the search stopped in a flat region.

The note is in the "note" column of the result (simplydata 3fd3b85 and later): it must say that the search may
have stopped in a flat region (recorded as 1 against the expected 1).

In [4]:
far = read_row(load("nist_BoxBOD").fit_formula(NIST_FORMULAS["BoxBOD"], start={"b1": 1, "b2": 1}))
checks.check("NIST BoxBOD", "fit_formula (start 1)", "says not converged when the fit is wrong (1 = yes)",
             1.0 if far["converged"] == "no" else 0.0)
checks.check("NIST BoxBOD", "fit_formula (start 1)", "note says the search stopped in a flat region (1 = yes)",
             1.0 if "flat region" in far["note"] else 0.0)

Fitted "y = b1*(1-exp(-b2*x))" to 6 rows: b1 = 172 (95 % CI infinite to infinite), b2 = 111 (95 % CI infinite to infinite). Residual standard error 49.4 on 4 degrees of freedom; R squared = 0.00 (only a rough guide for a curve).
Values found: b1, b2. Data: x. Method: nonlinear least squares (Levenberg-Marquardt), as R nls(); intervals are estimate +/- t x standard error (R confint() on nls profiles the likelihood and can differ a little).
Note: the search may have stopped in a flat region, where changing the values hardly changes the fit: R squared is 0 and there are no standard errors, so this is probably not the best fit. Try other start values (start={"b1": ..., "b2": ...}), or limits with between= and search="whole range".
agrees  NIST BoxBOD | fit_formula (start 1) | says not converged when the fit is wrong (1 = yes): simplydata 1, expected (A2) 1; difference 0.00e+00 (allowed 0e+00)
      Source: expected (A2) (expected) - FINDINGS A2: fit_formula must report converged = no when 

1

## Puromycin: a Michaelis-Menten curve with `table.fit_formula`

Reaction velocity against substrate concentration for the enzyme treated with Puromycin (12 rows). Bates & Watts
(1988) printed Vm = 212.7 and K = 0.06412. The same underneath: SciPy `curve_fit`.

In [5]:
found = read_row(load("puromycin").fit_formula("rate = Vm*conc/(K+conc)", start={"Vm": 200, "K": 0.1}))
checks.check_values("Puromycin (treated)", "fit_formula", {"Vm": found["Vm"], "K": found["K"]})

Fitted "rate = Vm*conc/(K+conc)" to 12 rows: Vm = 213 (95 % CI 197 to 228), K = 0.0641 (95 % CI 0.0457 to 0.0826). Residual standard error 10.9 on 10 degrees of freedom; R squared = 0.96 (only a rough guide for a curve).
Values found: Vm, K. Data: conc. Method: nonlinear least squares (Levenberg-Marquardt), as R nls(); intervals are estimate +/- t x standard error (R confint() on nls profiles the likelihood and can differ a little).
agrees  Puromycin (treated) | fit_formula | Vm: simplydata 212.68374208, Bates & Watts (1988), printed 212.7; difference 1.63e-02 (allowed 5e-02)
      Source: Bates & Watts (1988), printed (printed) - https://doi.org/10.1002/9780470316757
agrees  Puromycin (treated) | fit_formula | K: simplydata 0.0641212799903, Bates & Watts (1988), printed 0.06412; difference 1.28e-06 (allowed 5e-06)
      Source: Bates & Watts (1988), printed (printed) - https://doi.org/10.1002/9780470316757


2

## Bliss's beetles: a dose-response curve

Beetles killed out of those exposed at 8 doses of carbon disulphide (log10 mg per litre). Logistic regression on log
dose, `table.explain(outcome="yes or no", out_of=)`; Dobson & Barnett (2008) printed the fit -60.717 + 34.270 x (so the
tolerance is half a unit in the third decimal). The same underneath: statsmodels `GLM(Binomial)` on (killed,
survived).

The dose that kills half (LD50), 10^(60.717 / 34.270) from the printed fit, is read from
`table.fit_dose_response(outcome="yes or no", out_of=)` on the dose itself: its 2 parameter log-logistic curve (R
`drc::drm(type = "binomial", fct = LL.2())`) is the same model as the logistic regression on log dose. It is also
compared with R `glm()`'s 10^(-intercept / slope), to a relative difference of 1e-6.

In [6]:
bliss = load("bliss")
found = read_rows_by(bliss.explain("killed", using=["log10_dose"], outcome="yes or no", out_of="beetles"), "term")
checks.check_values("Bliss beetles", "explain (yes or no, out_of)",
                    {"intercept": found["(intercept)"]["log odds ratio"], "slope": found["log10_dose"]["log odds ratio"]})

Explaining "killed" from "log10_dose" (8 rows). Each odds ratio below holds the other columns fixed:
  intercept: 4.27e-27, the odds of yes when every number column is 0.
  log10_dose: each 1 more "log10_dose" multiplies the odds of yes by 7.65e+14 (95 % CI 2.54e+12 to 2.3e+17); fewer than 1 in 1000 studies would show this much if there were truly none (p < 0.0001).
Method: logistic regression, as R glm(cbind(killed, beetles - killed) ~ log10_dose, family = binomial); counted as yes: the number in "killed" out of "beetles" tries. Wald 95 % intervals and z-tests, as R confint.default() and summary().
Residual deviance 11.2 on 6 df (with no columns: 284 on 7 df). The model as a whole (likelihood-ratio chi-square = 273, df = 1): fewer than 1 in 1000 studies would show this much if none of the columns truly mattered (p < 0.0001).
How well it fits: pseudo R squared 0.879 (0 = no better than no columns). AIC 41.4, BIC 41.6: smaller is better, but only to compare models of the same rows (comp

2

In [7]:
doses = read_rows_by(bliss.fit_dose_response(x="dose", y="killed", outcome="yes or no", out_of="beetles"), "term")
checks.check("Bliss beetles", "fit_dose_response (yes or no, out_of)", "LD50", doses["ED50"]["estimate"])

Fitted a 2 parameter log-logistic, from 0 to 100 % (R LL.2) dose-response curve of "killed" out of "beetles" against the dose "dose" (8 rows).
The dose at which 50 % answer yes (ED50; LD50 when yes is death) is 59.1 (95 % CI 58.1 to 60.1).
Dose effect: If the dose truly had no effect, a change with dose this big would turn up in fewer than 1 in 1000 studies like this one (p < 0.0001). (drc::noEffect, a likelihood-ratio test)
Method: R drc::drm(killed / beetles ~ dose, weights = beetles, type = "binomial", fct = LL.2(), control = drc::drmc(relTol = 1e-15, maxIt = 10000)) (Ritz et al. 2015); drc::ED(m, c(50), interval = "delta"). simplydata chooses the 2 parameter log-logistic curve (from 0 to 100 %) and delta-method intervals (drc's ED() gives none unless asked).
agrees  Bliss beetles | fit_dose_response (yes or no, out_of) | LD50: simplydata 59.1181877104, Dobson & Barnett (2008), printed fit 59.1186543856; difference 4.67e-04 (allowed 1e-02)
      Source: Dobson & Barnett (2008), prin

2

## Result

In [8]:
checks.finish()

06_curves_and_dose_response: 542 comparisons - 459 agrees, 10 explained difference – machine, 73 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,06_curves_and_dose_response,NIST Misra1a,fit_formula (start 1),b1,238.942129,238.942129,NIST certified,1.084122e-07,6.000000,lre,9.343215e+00,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/nls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
1,06_curves_and_dose_response,NIST Misra1a,fit_formula (start 1),b1 standard error,2.707008,2.707008,NIST certified,1.000826e-07,6.000000,lre,7.432131e+00,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/nls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
2,06_curves_and_dose_response,NIST Misra1a,fit_formula (start 1),b2,0.000550,0.000550,NIST certified,-2.970615e-13,6.000000,lre,9.267640e+00,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/nls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
3,06_curves_and_dose_response,NIST Misra1a,fit_formula (start 1),b2 standard error,0.000007,0.000007,NIST certified,3.113081e-13,6.000000,lre,7.368157e+00,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/nls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
4,06_curves_and_dose_response,NIST Misra1a,fit_formula (start 1),residual standard deviation,0.101879,0.101879,NIST certified,2.435399e-12,6.000000,lre,1.062151e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/nls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537,06_curves_and_dose_response,Puromycin (treated),fit_formula,K,0.064121,0.064120,"Bates & Watts (1988), printed",1.279990e-06,0.000005,absolute,1.279990e-06,PASS,,agrees,,,,printed,https://doi.org/10.1002/9780470316757,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
538,06_curves_and_dose_response,Bliss beetles,"explain (yes or no, out_of)",intercept,-60.717455,-60.717000,"Dobson & Barnett (2008), printed",-4.545616e-04,0.000500,absolute,4.545616e-04,PASS,,agrees,,,,printed,"Dobson & Barnett (2008), An Introduction to Ge...","GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
539,06_curves_and_dose_response,Bliss beetles,"explain (yes or no, out_of)",slope,34.270326,34.270000,"Dobson & Barnett (2008), printed",3.257341e-04,0.000500,absolute,3.257341e-04,PASS,,agrees,,,,printed,"Dobson & Barnett (2008), An Introduction to Ge...","GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
540,06_curves_and_dose_response,Bliss beetles,"fit_dose_response (yes or no, out_of)",LD50,59.118188,59.118654,"Dobson & Barnett (2008), printed fit",-4.666752e-04,0.010000,absolute,4.666752e-04,PASS,,agrees,,,,printed,"Dobson & Barnett (2008), An Introduction to Ge...","GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
